# Notebook 04 — TCN (Temporal Convolutional Network) Training

The TCN replaces recurrent connections with **stacked dilated convolutions**, giving it a much larger receptive field per layer than a standard CNN — without the vanishing-gradient problems of RNNs.

**Architecture:**
- 9 dilated residual blocks, dilation = 1, 2, 4, 8, …, 256
- Each block: two weight-normed Conv1d layers + BatchNorm + ReLU + skip connection
- Non-causal (symmetric) padding — the full crossing signal is available at inference time
- Output head: 1×1 Conv1d → logits of shape `[B, 1300]`

Goals of this notebook:
1. Inspect architecture and verify receptive field covers the full 1,300-sample sequence
2. Sanity check: overfit on 10 samples
3. Full training run / load best checkpoint
4. Plot training curves
5. Visual inspection of predictions on 6 test examples


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

from src.dataset  import build_datasets
from src.models   import AxleTCN
from src.train    import train, train_one_epoch, build_loss
from src.evaluate import evaluate_model, print_metrics, pulses_to_peaks

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

JSON_PATH = '../axle_data.json/axle_data.json'
CKPT_DIR  = '../checkpoints'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED      = 42
print(f'Using device: {DEVICE}')


: 

## 1. Architecture overview

The **receptive field** of a stacked dilated TCN is:

$$RF = 1 + \sum_{i=0}^{N-1} 2 \cdot (k-1) \cdot 2^{i} = 1 + 2(k-1)(2^N - 1)$$

With $k=3$ (kernel size) and $N=9$ blocks: $RF = 1 + 2 \cdot 2 \cdot (512-1) = \mathbf{2045}$ samples — well over the 1,300-sample sequence.


In [ ]:
model = AxleTCN()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Architecture : AxleTCN")
print(f"Blocks       : {len(model.network)}")
print(f"Channels     : 64")
print(f"Kernel size  : 3 (symmetric padding)")
print(f"Dilations    : {[2**i for i in range(len(model.network))]}")
print(f"Receptive field : {model.receptive_field()} samples  (sequence length = 1300)")
print(f"Trainable params: {total_params:,}  ({total_params/1e6:.2f} M)")

# Verify output shape
x_test = torch.randn(4, 1, 1300)
y_test = model(x_test)
print(f"\nForward pass: input {list(x_test.shape)} → output {list(y_test.shape)}")


### Receptive field diagram

Each row shows one block's dilation. Shaded cells = samples that influence the central output at position 650.


In [ ]:
num_blocks  = 9
kernel_size = 3
seq_len     = 1300
center      = seq_len // 2

fig, ax = plt.subplots(figsize=(14, 4))

cumulative_rf = 1
colors = plt.cm.Blues(np.linspace(0.3, 0.9, num_blocks))

for i in range(num_blocks):
    dilation = 2 ** i
    block_rf = 2 * (kernel_size - 1) * dilation
    left  = max(0, center - cumulative_rf - block_rf // 2)
    right = min(seq_len, center + cumulative_rf + block_rf // 2)
    ax.barh(i, right - left, left=left, height=0.7,
            color=colors[i], edgecolor='white', linewidth=0.5)
    ax.text(2, i, f'Block {i+1}  (dilation={dilation:3d}, RF contribution=±{block_rf//2})',
            va='center', fontsize=8, color='#333')
    cumulative_rf += block_rf

ax.axvline(center, color='#e74c3c', linewidth=1.5, linestyle='--', label='Query position (sample 650)')
ax.set_xlim(0, seq_len)
ax.set_yticks(range(num_blocks))
ax.set_yticklabels([f'Block {i+1}' for i in range(num_blocks)])
ax.set_xlabel('Sample index (of 1300)')
ax.set_title('TCN receptive field growth across blocks')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Final receptive field: {model.receptive_field()} samples (covers full 1300-sample sequence)')


## 2. Sanity check — overfit on 10 samples

If the model can't memorise 10 examples, something is broken. Loss should reach near-zero.


In [ ]:
torch.manual_seed(SEED)
train_ds, val_ds, test_ds = build_datasets(JSON_PATH, seed=SEED)

tiny_ds     = Subset(train_ds, list(range(10)))
tiny_loader = DataLoader(tiny_ds, batch_size=10, shuffle=False)

tiny_model = AxleTCN().to(DEVICE)
criterion_plain = torch.nn.BCEWithLogitsLoss()
optimizer       = torch.optim.AdamW(tiny_model.parameters(), lr=5e-3)

losses = []
for epoch in range(1, 201):
    loss = train_one_epoch(tiny_model, tiny_loader, criterion_plain, optimizer, DEVICE, scaler=None)
    losses.append(loss)
    if epoch % 40 == 0:
        print(f'Epoch {epoch:3d}  loss={loss:.6f}')

print('✓ Sanity check PASSED' if losses[-1] < 0.01 else f'WARNING: final loss={losses[-1]:.4f}')

plt.figure(figsize=(7, 3))
plt.plot(losses, color='seagreen', linewidth=1.2)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss (plain)')
plt.title('Sanity check — TCN overfitting on 10 samples')
plt.yscale('log'); plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()


## 3. Full training run

For the thesis, training was performed from the CLI:
```
python -m src.train --model tcn --epochs 100 --batch_size 64 --patience 25
```
Best checkpoint was saved at **epoch 8** (val F1 = 0.9982). Training converged at epoch 33.

The cell below loads the saved checkpoint and evaluates on the test set.  
*(To re-run training from scratch, uncomment the `train(...)` call — takes ~3–4 hours on GPU.)*


In [ ]:
# --- Uncomment to re-run training from scratch ---
# test_metrics = train(
#     model_name='tcn', json_path=JSON_PATH, epochs=100,
#     batch_size=64, lr=1e-3, patience=25, num_workers=0,
#     seed=SEED, checkpoint_dir=CKPT_DIR,
# )

# Load best checkpoint
ckpt  = torch.load(os.path.join(CKPT_DIR, 'tcn_best.pt'), map_location=DEVICE, weights_only=False)
model = AxleTCN().to(DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded checkpoint — epoch {ckpt['epoch']},  val F1 = {ckpt['val_f1']:.4f}")

# Test-set evaluation
_, _, test_ds = build_datasets(JSON_PATH, seed=SEED)
test_loader   = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
test_metrics  = evaluate_model(model, test_loader, DEVICE, threshold=0.5, tolerance=5)
print_metrics(test_metrics, prefix='TCN — Test set')


## 4. Training curves


In [ ]:
log_path = os.path.join(CKPT_DIR, 'tcn_log.csv')
epochs_list, train_losses, val_losses, val_f1s = [], [], [], []

with open(log_path) as f:
    for row in csv.DictReader(f):
        epochs_list.append(int(row['epoch']))
        train_losses.append(float(row['train_loss']))
        val_losses.append(float(row['val_loss']))
        val_f1s.append(float(row['val_f1']))

best_ep = epochs_list[int(np.argmax(val_f1s))]
best_f1 = max(val_f1s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_list, train_losses, label='Train loss', color='#3498db', linewidth=1.5)
ax1.plot(epochs_list, val_losses,   label='Val loss',   color='seagreen', linewidth=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('TCN — Loss curves'); ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.5)

ax2.plot(epochs_list, val_f1s, color='seagreen', marker='.', markersize=4, linewidth=1.5)
ax2.axvline(best_ep, color='#e74c3c', linestyle='--', linewidth=1.2,
            label=f'Best epoch {best_ep}  (F1={best_f1:.4f})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val F1')
ax2.set_title('TCN — Validation F1'); ax2.legend(fontsize=9)
ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()
print(f'Best: epoch {best_ep}, val F1 = {best_f1:.4f}')


## 5. Visual inspection on 6 test examples

Blue = raw signal. Orange = TCN probability (scaled to signal range). Green lines = ground truth. Red dashed = TCN predictions.


In [ ]:
model.eval()
np.random.seed(SEED)
sample_idx = np.random.choice(len(test_ds), 6, replace=False)

fig, axes = plt.subplots(6, 1, figsize=(14, 3 * 6))
for ax, idx in zip(axes, sample_idx):
    sig_t, pulse_t = test_ds[idx]
    with torch.no_grad():
        logit = model(sig_t.unsqueeze(0).to(DEVICE))
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()

    sig        = sig_t.squeeze(0).numpy()
    true_peaks = pulses_to_peaks(pulse_t.numpy(), threshold=0.5)
    pred_peaks = pulses_to_peaks(prob,             threshold=0.5)

    # Scale probability to signal amplitude range for overlay
    prob_scaled = prob * (sig.max() - sig.min()) + sig.min()

    ax.plot(sig,        color='steelblue',  linewidth=0.8, alpha=0.7, label='Signal')
    ax.plot(prob_scaled, color='darkorange', linewidth=0.9, alpha=0.8, label='TCN prob (scaled)')
    ax.vlines(true_peaks, sig.min(), sig.max(), colors='#2ecc71',
              linewidth=1.8, label=f'Ground truth ({len(true_peaks)})')
    ax.vlines(pred_peaks, sig.min(), sig.max(), colors='#e74c3c',
              linewidth=1.2, linestyle='--', label=f'TCN pred ({len(pred_peaks)})')

    match = len(true_peaks) == len(pred_peaks)
    ax.set_title(f'Test record {idx}  —  {"✓" if match else "✗"} '
                 f'({len(true_peaks)} axles)', fontsize=9)
    ax.legend(loc='upper right', fontsize=7)

axes[-1].set_xlabel('Sample index')
plt.tight_layout()
plt.show()
